In [1]:
!pip install scikit-learn --quiet
!pip install xgboost --quiet
!pip install imbalanced-learn --quiet

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV
import time

In [3]:
# Load engineered features
df = pd.read_csv('fraud_features_ready.csv')

# Separate features and target
X = df.drop('isFraud', axis=1)
y = df['isFraud']

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} transactions")
print(f"Test set: {len(X_test)} transactions")
print(f"\nTraining fraud rate: {y_train.mean()*100:.3f}%")
print(f"\n✅ X_train is DataFrame: {isinstance(X_train, pd.DataFrame)}")
print(f"✅ Feature names: {X_train.columns.tolist()}")

# Scale features (important for neural networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures scaled!")

Training set: 2216327 transactions
Test set: 554082 transactions

Training fraud rate: 0.296%

✅ X_train is DataFrame: True
✅ Feature names: ['step', 'amount', 'origin_balance_change', 'destination_balance_change', 'origin_error', 'destination_error', 'origin_zero_after', 'destination_zero_before', 'amount_to_origin_balance', 'amount_to_destination_balance', 'origin_out_degree', 'destination_in_degree', 'origin_pagerank', 'destination_pagerank', 'velocity']

Features scaled!


# Class Imbalance

In [4]:
# Fraud is 0.13% - very imbalanced!
# SMOTE creates synthetic fraud examples

print("Balancing dataset with SMOTE...")
smote = SMOTE(random_state=42, sampling_strategy=0.3)  # Make frauds 30% of dataset
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"Original training set: {len(X_train)} transactions")
print(f"Balanced training set: {len(X_train_balanced)} transactions")
print(f"New fraud rate: {y_train_balanced.mean()*100:.1f}%")

Balancing dataset with SMOTE...
Original training set: 2216327 transactions
Balanced training set: 2872684 transactions
New fraud rate: 23.1%


# Train Models

In [5]:
# Model 1: Random Forest
print("\n" + "="*50)
print("Training Random Forest...")
start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_balanced, y_train_balanced)

print(f"Training time: {time.time()-start:.2f} seconds")

# Predict
y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate
print("\nRandom Forest Results:")
print(classification_report(y_test, y_pred_rf, target_names=['Normal', 'Fraud']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")

# Model 2: XGBoost
print("\n" + "="*50)
print("Training XGBoost...")
start = time.time()

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    random_state=42,
    eval_metric='auc'
)
xgb_model.fit(X_train_balanced, y_train_balanced)

print(f"Training time: {time.time()-start:.2f} seconds")

# Predict
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_pred_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate
print("\nXGBoost Results:")
print(classification_report(y_test, y_pred_xgb, target_names=['Normal', 'Fraud']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_xgb):.4f}")


Training Random Forest...
Training time: 278.34 seconds

Random Forest Results:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00    552439
       Fraud       0.91      1.00      0.95      1643

    accuracy                           1.00    554082
   macro avg       0.96      1.00      0.98    554082
weighted avg       1.00      1.00      1.00    554082

ROC-AUC Score: 0.9991

Training XGBoost...
Training time: 19.12 seconds

XGBoost Results:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00    552439
       Fraud       0.48      0.99      0.65      1643

    accuracy                           1.00    554082
   macro avg       0.74      0.99      0.82    554082
weighted avg       1.00      1.00      1.00    554082

ROC-AUC Score: 0.9994


# Finetuning XGBOOST

In [7]:
import numpy as np
# Step 1: SMOTE on UNSCALED data
print("Applying SMOTE...")
smote = SMOTE(random_state=42, sampling_strategy=0.3)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"Balanced training set: {len(X_train_bal)} transactions")
print(f"New fraud rate: {y_train_bal.mean()*100:.1f}%")

# Step 2: Scale the BALANCED data
print("\nScaling balanced data...")
scaler = StandardScaler()
X_train_bal_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Step 3: Calculate scale_pos_weight for imbalanced data
scale = len(y_train[y_train==0]) / len(y_train[y_train==1])

# Step 4: Hyperparameter tuning
print("\nStarting hyperparameter tuning...")
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    scale_pos_weight=scale,
    random_state=42
)

param_dist = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6, 8],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'subsample': [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'n_estimators': [200, 400, 600]
}

search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1_macro',
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train_bal_scaled, y_train_bal)  # ← Train on SCALED data
best_xgb = search.best_estimator_

# Step 5: Evaluate on SCALED test data
print("\n" + "="*50)
print("Best Model Performance:")
print("="*50)
y_pred = best_xgb.predict(X_test_scaled)  # ← Predict on SCALED data
y_pred_proba = best_xgb.predict_proba(X_test_scaled)[:, 1]

print("Best Parameters:", search.best_params_)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print(f"\nROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

# Step 6: Test with your fraud case
print("\n" + "="*50)
print("Testing with Known Fraud Case:")
print("="*50)

# Create test case as DataFrame WITH column names
test_fraud = pd.DataFrame({
    'step': [0],
    'amount': [5000000],
    'origin_balance_change': [5000000],
    'destination_balance_change': [5000000],
    'origin_error': [0],
    'destination_error': [0],
    'origin_zero_after': [1],
    'destination_zero_before': [1],
    'amount_to_origin_balance': [1.0],
    'amount_to_destination_balance': [5000000],
    'origin_out_degree': [1],
    'destination_in_degree': [3],
    'origin_pagerank': [0.001],
    'destination_pagerank': [0.001],
    'velocity': [0.5]
})

# Transform
test_fraud_scaled = scaler.transform(test_fraud)

# Predict
fraud_prob = best_xgb.predict_proba(test_fraud_scaled)[0][1]

print(f"Amount: ₦5,000,000")
print(f"Origin Balance: ₦5,000,000 (will be emptied)")
print(f"Dest Balance: ₦0 (new account)")
print(f"Dest Connections: 3")
print(f"\nFraud Probability: {fraud_prob*100:.2f}%")
print(f"Expected: >80%")
print(f"Status: {'✅ CORRECT' if fraud_prob > 0.8 else '❌ STILL WRONG'}")

Applying SMOTE...
Balanced training set: 2872684 transactions
New fraud rate: 23.1%

Scaling balanced data...

Starting hyperparameter tuning...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Best Model Performance:
Best Parameters: {'subsample': 0.6, 'n_estimators': 600, 'min_child_weight': 7, 'max_depth': 5, 'learning_rate': 0.2, 'gamma': 0, 'colsample_bytree': 0.6}

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    552439
           1       0.48      0.99      0.65      1643

    accuracy                           1.00    554082
   macro avg       0.74      0.99      0.82    554082
weighted avg       1.00      1.00      1.00    554082


ROC-AUC: 0.9983

Testing with Known Fraud Case:
Amount: ₦5,000,000
Origin Balance: ₦5,000,000 (will be emptied)
Dest Balance: ₦0 (new account)
Dest Connections: 3

Fraud Probability: 100.00%
Expected: >80%
Status: ✅ CORRECT


In [8]:


import joblib
joblib.dump(best_xgb, 'xgb_fraud_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("✅ Model saved: xgb_fraud_model.pkl")
print("✅ Scaler saved: scaler.pkl")
print("\nNow restart your Streamlit app!")

✅ Model saved: xgb_fraud_model.pkl
✅ Scaler saved: scaler.pkl

Now restart your Streamlit app!
